In [ ]:
# Feature extraction con CSP e epoch e classificazione binaria (riposo vs attivazione). 
# Classificatore SVM lineare. Le performance sono chiaramente migliori rispetto alle windows, ma ha poca importanza.
# Classificazione binaria tra sinistra e destra
import mne
from mne.decoding import CSP
from mne_bids import BIDSPath, read_raw_bids
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from pathlib import Path


mne.set_log_level('WARNING')

root = "../data"
root = (Path(root).resolve())
runs = ["4", "8", "12"]   # Le run che vengono prese in considerazione

X_all = []  # Vettore X con tutti i dati di tutte le finestre
y_all = []  # Vettore y con tutte le etichette di tutte le finestre

scaler = StandardScaler()

# Eseguiamo la scansione di tutti i soggetti 
for i in range(1, 3):
    subject = f"{i:03d}"
    # Per ogni soggetto eseguiamo la scansione sulle run di nostro interesse
    for run in runs:
        bids_path = BIDSPath(   # Specifichiamo il percorso del dataset e BIDS eseguirà correttamente la scansione 
            subject=subject,
            task="motion",
            run=run,
            datatype="eeg",
            root=root,
        )
        
        try:
            # Fase 1: lettura dei dati e pre-processing
            raw = read_raw_bids(bids_path, verbose=False)  
            events, event_id = mne.events_from_annotations(raw, verbose=False) 
            raw.load_data(verbose=False) # Carico i dati in memoria per poter filtrare ecc.
            raw.filter(l_freq=8, h_freq=30, verbose=False)  # Filtro passa banda 1-30 Hz
            #raw.set_eeg_reference('average', projection=False, verbose=False)  # Riferimento medio
        
            
            epochs = mne.Epochs(
                raw,
                events,
                event_id=event_id,
                tmin=0.0,
                tmax=4.0,
                baseline=(0.0, 0.2),
                preload=True,
                verbose=False
            )          
            # ar = AutoReject()

            # Addestro il modello solo su distizione destra/sinistra, escludendo il riposo
            epochs = epochs[epochs.events[:, 2] != 1]
            y_binary = np.where(epochs.events[:, 2] == 2, 0, 1)
            X = epochs.get_data()

            # Standardizzazione per soggetto
            n_epochs, n_channels, n_times = X.shape
            X_flat = X.reshape(n_epochs, n_channels * n_times)
            X_scaled_flat = scaler.fit_transform(X_flat)
            X_scaled = X_scaled_flat.reshape(n_epochs, n_channels, n_times)
            # ---------------------------------------

            X_all.append(X_scaled)  
            y_all.append(y_binary)      
            print(f"Soggetto {subject} run {run} - Campioni: {X.shape[0]}, Feature per campione: {X.shape[1]}, Etichette: {y_binary.shape[0]}")


        except Exception as e:
            print(f"Errore {subject}: {e}")



Soggetto 001 run 4 - Campioni: 15, Feature per campione: 64, Etichette: 15
Soggetto 001 run 8 - Campioni: 15, Feature per campione: 64, Etichette: 15
Soggetto 001 run 12 - Campioni: 15, Feature per campione: 64, Etichette: 15
Soggetto 002 run 4 - Campioni: 15, Feature per campione: 64, Etichette: 15
Soggetto 002 run 8 - Campioni: 15, Feature per campione: 64, Etichette: 15
Soggetto 002 run 12 - Campioni: 15, Feature per campione: 64, Etichette: 15


In [10]:
# Concatenazione finale
from sklearn.model_selection import GridSearchCV


X_all = np.vstack(X_all)
y_all = np.concatenate(y_all)

# Pipeline: scaling + LDA
pipe = make_pipeline(
    CSP(log=True), 
    SVC()
)
# plt.scatter(
#     X_all[:,0],
#     X_all[:,1],
#     c=y_all
# )

# plt.xlabel("CSP 1")
# plt.ylabel("CSP 2")
# plt.show()

param_grid = {
    'csp__n_components': [ 6, 8],
    'svc__kernel': ['rbf'],
    'svc__C': [10, 100],
    'svc__gamma': ['scale', 'auto', 0.01, 0.1]
}

# Cross-validation stratificata
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipe, 
    param_grid, 
    cv=cv_strategy, 
    scoring='balanced_accuracy', # Consigliata per BCI se le classi non sono perfettamente bilanciate
    n_jobs=-1,                   # Utilizza tutti i core della CPU per velocizzare
    verbose=1
)

print("Inizio Grid Search...")
grid_search.fit(X_all, y_all)

print("-" * 30)
print(f"Miglior Balanced Accuracy: {grid_search.best_score_:.4f}")
print(f"Migliori parametri: {grid_search.best_params_}")
print("-" * 30)

Inizio Grid Search...
Fitting 5 folds for each of 16 candidates, totalling 80 fits
------------------------------
Miglior Balanced Accuracy: 0.7647
Migliori parametri: {'csp__n_components': 8, 'svc__C': 100, 'svc__gamma': 'auto', 'svc__kernel': 'rbf'}
------------------------------
